# Redes Neuronales III – Anexo: equivalente en PyTorch

**Caso de uso (sector salud):** clasificación de tumores de mama como benignos o malignos a partir de 30 características de núcleos celulares (dataset *Breast Cancer Wisconsin – Diagnostic*, 569 muestras). Uso exclusivamente académico: no sustituye un diagnóstico médico.

## 1. Datos (mismos que en Keras)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42

# 1. Carga: Breast Cancer Wisconsin (Diagnostic), incluido en scikit-learn
data = load_breast_cancer(as_frame=True)
X = data.data
y = (1 - data.target).astype("int32")      # 1 = maligno, 0 = benigno

# 2. Distribucion estratificada 60 / 20 / 20 (train / dev / test)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_dev, y_train, y_dev = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=SEED)

# 3. Preprocesamiento: estandarizacion ajustada SOLO con train (evita fuga de datos)
scaler = StandardScaler().fit(X_train)
Xtr_s, Xdv_s, Xte_s = [scaler.transform(a).astype("float32")
                       for a in (X_train, X_dev, X_test)]
Xtr_raw = X_train.to_numpy(dtype="float32")     # sin escalar (solo para comparar)
Xdv_raw = X_dev.to_numpy(dtype="float32")
N_FEATURES = Xtr_s.shape[1]

resumen = pd.DataFrame({
    "conjunto": ["train", "dev", "test"],
    "muestras": [len(y_train), len(y_dev), len(y_test)],
    "malignos": [int(y_train.sum()), int(y_dev.sum()), int(y_test.sum())]})
resumen["% malignos"] = (100 * resumen["malignos"] / resumen["muestras"]).round(1)
print(resumen.to_string(index=False))
print("Caracteristicas:", N_FEATURES)

## 2. Red, entrenamiento con Early Stopping y evaluación

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)

def loader(X, y, shuffle):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y.to_numpy(), dtype=torch.float32))
    return DataLoader(ds, batch_size=32, shuffle=shuffle)

train_dl, dev_dl = loader(Xtr_s, y_train, True), loader(Xdv_s, y_dev, False)

model = nn.Sequential(
    nn.Linear(N_FEATURES, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(32, 1)).to(device)                   # logits (sigmoide implicita en la perdida)

loss_fn = nn.BCEWithLogitsLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)   # L2 acoplada
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=30, gamma=0.5)    # LR decay

mejor, paciencia, esperando, mejores_pesos = float("inf"), 15, 0, None
for epoca in range(200):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = loss_fn(model(xb).squeeze(1), yb)
        loss.backward()
        opt.step()
    sched.step()

    model.eval(); val = 0.0
    with torch.no_grad():
        for xb, yb in dev_dl:
            val += loss_fn(model(xb.to(device)).squeeze(1), yb.to(device)).item() * len(yb)
    val /= len(dev_dl.dataset)

    if val < mejor:                                # Early stopping manual
        mejor, esperando = val, 0
        mejores_pesos = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        esperando += 1
        if esperando >= paciencia:
            print("Early stopping en la epoca", epoca + 1)
            break

model.load_state_dict(mejores_pesos); model.eval()
with torch.no_grad():
    p = torch.sigmoid(model(torch.tensor(Xte_s).to(device)).squeeze(1)).cpu().numpy()
y_pred = (p >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f"accuracy={accuracy_score(y_test, y_pred):.4f}  precision={precision_score(y_test, y_pred):.4f}  "
      f"recall={recall_score(y_test, y_pred):.4f}  especificidad={tn/(tn+fp):.4f}  f1={f1_score(y_test, y_pred):.4f}")
print("Matriz de confusion [[TN, FP], [FN, TP]]:", [[tn, fp], [fn, tp]])